# MATCHA-TTS Training Complete Pipeline for Kaggle
## Setup môi trường và training Matcha-TTS với Prosody Analysis (PhoBERT)

Notebook này tổng hợp toàn bộ code từ project để train trên Kaggle

In [ ]:
# 1. Clone repo từ GitHub & Checkout branch
!git clone https://github.com/Neyly112/IPIATTS.git
%cd IPIATTS
!git checkout ket_hop_LLM_prosody_PhoBert

# 2. Cài đặt dependencies
!pip install lightning==2.0.0 transformers>=4.30.0 phonemizer underthesea num2words librosa soundfile praat-parselmouth tensorboard einops
!pip install -r requirements.txt

# 3. Tạo output directory
import os
os.makedirs("/kaggle/working/outputs", exist_ok=True)
print("✅ Đã setup xong!")


## 2. Fix Lightning + Espeak

In [ ]:
# Fix lỗi Lightning
!pip uninstall -y lightning lightning-cloud
!pip install lightning==2.1.0 lightning-cloud==0.5.37

# Cài đặt espeak-ng
!apt-get update
!apt-get install -y espeak-ng
print("✅ Đã fix Lightning và espeak-ng!")


## 3. Cập nhật Symbols, Cleaners và DataModule

In [ ]:
# Cập nhật symbols.py với tiếng Việt đầy đủ
symbols_code = """
# -*- coding: utf-8 -*-

_pad = "_"
_punctuation = [
    chr(35),   # #
    chr(59),   # ;
    chr(58),   # :
    chr(44),   # ,
    chr(46),   # .
    chr(33),   # !
    chr(63),   # ?
    chr(161),  # ¡
    chr(191),  # ¿
    chr(45),   # -
    chr(8212), # —
    chr(8230), # …
    chr(39),   # '
    chr(34),   # "
    chr(171),  # «
    chr(187),  # »
    chr(8220), # "
    chr(8221), # "
    chr(40),   # (
    chr(41),   # )
    chr(91),   # [
    chr(93),   # ]
    chr(47),   # /
    chr(37),   # %
    chr(32),   # space
]

_letters_lower = list("abcdefghijklmnopqrstuvwxyzđ")
_vietnamese_accents_lower = [
    "à", "á", "ả", "ã", "ạ",
    "ă", "ằ", "ắ", "ẳ", "ẵ", "ặ",
    "â", "ầ", "ấ", "ẩ", "ẫ", "ậ",
    "è", "é", "ẻ", "ẽ", "ẹ",
    "ê", "ề", "ế", "ể", "ễ", "ệ",
    "ì", "í", "ỉ", "ĩ", "ị",
    "ò", "ó", "ỏ", "õ", "ọ",
    "ô", "ồ", "ố", "ổ", "ỗ", "ộ",
    "ơ", "ờ", "ớ", "ở", "ỡ", "ợ",
    "ù", "ú", "ủ", "ũ", "ụ",
    "ư", "ừ", "ứ", "ử", "ữ", "ự",
    "ỳ", "ý", "ỷ", "ỹ", "ỵ",
]

_letters_upper = list("ABCDEFGHIJKLMNOPQRSTUVWXYZĐ")
_vietnamese_accents_upper = [
    "À", "Á", "Ả", "Ã", "Ạ",
    "Ă", "Ằ", "Ắ", "Ẳ", "Ẵ", "Ặ",
    "Â", "Ầ", "Ấ", "Ẩ", "Ẫ", "Ậ",
    "È", "É", "Ẻ", "Ẽ", "Ẹ",
    "Ê", "Ề", "Ế", "Ể", "Ễ", "Ệ",
    "Ì", "Í", "Ỉ", "Ĩ", "Ị",
    "Ò", "Ó", "Ỏ", "Õ", "Ọ",
    "Ô", "Ồ", "Ố", "Ổ", "Ỗ", "Ộ",
    "Ơ", "Ờ", "Ớ", "Ở", "Ỡ", "Ợ",
    "Ù", "Ú", "Ủ", "Ũ", "Ụ",
    "Ư", "Ừ", "Ứ", "Ử", "Ữ", "Ự",
    "Ỳ", "Ý", "Ỷ", "Ỹ", "Ỵ",
]

_digits = list("0123456789")

_letters_ipa = [
    chr(593), chr(592), chr(594), chr(230), chr(595), chr(665), chr(946),
    chr(596), chr(597), chr(231), chr(599), chr(598), chr(240), chr(676),
    chr(601), chr(600), chr(602), chr(603), chr(604), chr(605), chr(606),
    chr(607), chr(644), chr(609), chr(608), chr(610), chr(667), chr(614),
    chr(615), chr(295), chr(613), chr(668), chr(616), chr(618), chr(669),
    chr(621), chr(620), chr(619), chr(622), chr(671), chr(625), chr(623),
    chr(624), chr(331), chr(627), chr(626), chr(628), chr(248), chr(629),
    chr(632), chr(952), chr(339), chr(630), chr(664), chr(633), chr(634),
    chr(638), chr(635), chr(640), chr(641), chr(637), chr(642), chr(643),
    chr(648), chr(649), chr(650), chr(651), chr(11377), chr(652), chr(611),
    chr(612), chr(653), chr(967), chr(654), chr(655), chr(657), chr(656),
    chr(658), chr(660), chr(673), chr(661), chr(674), chr(448), chr(449),
    chr(450), chr(451), chr(712), chr(716), chr(720), chr(721), chr(700),
    chr(692), chr(688), chr(689), chr(690), chr(695), chr(736), chr(740),
    chr(734), chr(8595), chr(8593), chr(8594), chr(8599), chr(8600),
    chr(809), chr(7547), chr(810), chr(865)
]

symbols = (
    [_pad]
    + _punctuation
    + _letters_lower
    + _letters_upper
    + _vietnamese_accents_lower
    + _vietnamese_accents_upper
    + _digits
    + _letters_ipa
)

SPACE_ID = symbols.index(" ")
_symbol_to_id = {s: i for i, s in enumerate(symbols)}
_id_to_symbol = {i: s for i, s in enumerate(symbols)}
"""

with open("matcha/text/symbols.py", "w", encoding="utf-8") as f:
    f.write(symbols_code)

print("✅ Đã cập nhật symbols.py!")


## 4. Cập nhật Cleaners (Training vs Inference)

In [ ]:
# Cập nhật cleaners.py cho TRAINING (input đã là IPA) và INFERENCE (text -> IPA)
cleaner_content = """
import re
import sys

try:
    from underthesea import text_normalize
    from phonemizer.backend import EspeakBackend
    from num2words import num2words
except ImportError:
    pass

_ESPEAK = None
try:
    _ESPEAK = EspeakBackend(
        "vi",
        preserve_punctuation=True,
        language_switch="remove-flags", 
        with_stress=True,               
        tie=True
    )
except Exception:
    pass

_RE_NUMBER = re.compile(r"\d+")

def basic_cleaners_phothong(text):
    '''
    Cleaner dùng cho TRAINING (input đã là IPA).
    Chỉ chuẩn hóa khoảng trắng, KHÔNG gọi espeak.
    '''
    if text is None: 
        return ""
    return re.sub(r"\s+", " ", text).strip()

def vietnamese_text_to_ipa(text):
    '''
    Cleaner dùng cho INFERENCE (text thường -> IPA).
    '''
    if _ESPEAK is None:
        raise RuntimeError("Lỗi: Chưa cài espeak-ng hoặc phonemizer!")

    text = text_normalize(text)
    
    def _replace_number(match):
        try:
            return num2words(int(match.group()), lang='vi')
        except:
            return match.group()
            
    text = _RE_NUMBER.sub(_replace_number, text)
    text = text.replace("%", " phần trăm").replace("&", " và ").replace("+", " cộng ")
    ipa = _ESPEAK.phonemize(text, strip=True)
    ipa = re.sub(r'\(.*?\)', '', ipa)
    ipa = re.sub(r"\s+", " ", ipa).strip()
    
    return ipa
"""

with open("matcha/text/cleaners.py", "w", encoding="utf-8") as f:
    f.write(cleaner_content)

print("✅ Đã cập nhật cleaners.py!")


## 5. Cập nhật TextMelDataModule

In [ ]:
# Cập nhật TextMelDataModule để đọc 3 cột và tính thêm pitch/energy cho prosody
# Raw_text sẽ được dùng cho PhoBERT (Prosody Analysis)

datamodule_code = """
import random
from pathlib import Path
from typing import Optional
import torch
import torchaudio as ta
from torch.nn import functional as F
from lightning import LightningDataModule
from torch.utils.data.dataloader import DataLoader
from matcha.text import text_to_sequence
from matcha.utils.audio import mel_spectrogram
from matcha.utils.model import fix_len_compatibility, normalize
from matcha.utils.utils import intersperse


def parse_filelist(filelist_path, split_char="|"):
    with open(filelist_path, encoding="utf-8") as f:
        filepaths_and_text = [line.strip().split(split_char) for line in f]
    return filepaths_and_text


def pad_or_trim(feature: torch.Tensor, target_len: int) -> torch.Tensor:
    # Pad hoặc cắt feature 1D về đúng số frame mel
    if feature.shape[-1] < target_len:
        feature = F.pad(feature, (0, target_len - feature.shape[-1]))
    elif feature.shape[-1] > target_len:
        feature = feature[..., :target_len]
    return feature


class TextMelDataModule(LightningDataModule):
    def __init__(
        self,
        name,
        train_filelist_path,
        valid_filelist_path,
        batch_size,
        num_workers,
        pin_memory,
        cleaners,
        add_blank,
        n_spks,
        n_fft,
        n_feats,
        sample_rate,
        hop_length,
        win_length,
        f_min,
        f_max,
        data_statistics,
        seed,
        load_durations,
        audio_root=None,
    ):
        super().__init__()
        self.save_hyperparameters(logger=False)

    def setup(self, stage: Optional[str] = None):
        self.trainset = TextMelDataset(
            self.hparams.train_filelist_path,
            self.hparams.n_spks,
            self.hparams.cleaners,
            self.hparams.add_blank,
            self.hparams.n_fft,
            self.hparams.n_feats,
            self.hparams.sample_rate,
            self.hparams.hop_length,
            self.hparams.win_length,
            self.hparams.f_min,
            self.hparams.f_max,
            self.hparams.data_statistics,
            self.hparams.seed,
            self.hparams.load_durations,
            audio_root=self.hparams.audio_root,
        )
        self.validset = TextMelDataset(
            self.hparams.valid_filelist_path,
            self.hparams.n_spks,
            self.hparams.cleaners,
            self.hparams.add_blank,
            self.hparams.n_fft,
            self.hparams.n_feats,
            self.hparams.sample_rate,
            self.hparams.hop_length,
            self.hparams.win_length,
            self.hparams.f_min,
            self.hparams.f_max,
            self.hparams.data_statistics,
            self.hparams.seed,
            self.hparams.load_durations,
            audio_root=self.hparams.audio_root,
        )

    def train_dataloader(self):
        return DataLoader(
            self.trainset,
            batch_size=self.hparams.batch_size,
            num_workers=self.hparams.num_workers,
            pin_memory=self.hparams.pin_memory,
            shuffle=True,
            collate_fn=TextMelBatchCollate(self.hparams.n_spks),
        )

    def val_dataloader(self):
        return DataLoader(
            self.validset,
            batch_size=self.hparams.batch_size,
            num_workers=self.hparams.num_workers,
            pin_memory=self.hparams.pin_memory,
            shuffle=False,
            collate_fn=TextMelBatchCollate(self.hparams.n_spks),
        )


class TextMelDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        filelist_path,
        n_spks,
        cleaners,
        add_blank=True,
        n_fft=1024,
        n_mels=80,
        sample_rate=22050,
        hop_length=256,
        win_length=1024,
        f_min=0.0,
        f_max=8000,
        data_parameters=None,
        seed=None,
        load_durations=False,
        audio_root=None,
    ):
        self.filepaths_and_text = parse_filelist(filelist_path)
        self.n_spks = n_spks
        self.cleaners = cleaners
        self.add_blank = add_blank
        self.n_fft = n_fft
        self.n_mels = n_mels
        self.sample_rate = sample_rate
        self.hop_length = hop_length
        self.win_length = win_length
        self.f_min = f_min
        self.f_max = f_max
        self.load_durations = load_durations
        self.audio_root = Path(audio_root) if audio_root else None
        self.data_parameters = data_parameters if data_parameters else {
            "mel_mean": 0,
            "mel_std": 1,
            "pitch_mean": 0,
            "pitch_std": 1,
            "energy_mean": 0,
            "energy_std": 1,
        }
        random.seed(seed)
        random.shuffle(self.filepaths_and_text)

    def load_audio(self, filepath: str) -> torch.Tensor:
        filepath = Path(filepath)
        if self.audio_root and not filepath.is_absolute():
            filepath = self.audio_root / filepath
        audio, sr = ta.load(filepath)
        if sr != self.sample_rate:
            audio = ta.functional.resample(audio, sr, self.sample_rate)
        if audio.shape[0] > 1:
            audio = audio.mean(dim=0, keepdim=True)
        return audio

    def get_datapoint(self, filepath_and_text):
        if len(filepath_and_text) >= 3:
            filepath = filepath_and_text[0]
            raw_text = filepath_and_text[1]
            ipa_text = filepath_and_text[2]
        else:
            filepath = filepath_and_text[0]
            raw_text = filepath_and_text[1]
            ipa_text = filepath_and_text[1]

        text, cleaned_text = self.get_text(ipa_text, add_blank=self.add_blank)
        audio = self.load_audio(filepath)
        mel = self.get_mel(audio)
        pitch, energy = self.get_pitch_energy(audio, mel.shape[-1])
        durations = self.get_durations(filepath, text) if self.load_durations else None

        return {
            "x": text,
            "y": mel,
            "pitch": pitch,
            "energy": energy,
            "spk": 0,
            "filepath": filepath,
            "raw_text": raw_text,
            "durations": durations,
        }

    def get_mel(self, audio: torch.Tensor):
        mel = mel_spectrogram(
            audio,
            self.n_fft,
            self.n_mels,
            self.sample_rate,
            self.hop_length,
            self.win_length,
            self.f_min,
            self.f_max,
            center=False,
        ).squeeze()
        mel = normalize(mel, self.data_parameters.get("mel_mean", 0), self.data_parameters.get("mel_std", 1))
        return mel

    def get_pitch_energy(self, audio: torch.Tensor, target_len: int):
        mono = audio
        # Fix: detect_pitch_frequency không có hop_length parameter, dùng frame_time
        frame_time = self.hop_length / self.sample_rate
        pitch = ta.functional.detect_pitch_frequency(
            mono,
            sample_rate=self.sample_rate,
            frame_time=frame_time,
        ).squeeze(0)
        pitch = torch.log1p(pitch)
        pitch = torch.nan_to_num(pitch, nan=0.0, posinf=0.0, neginf=0.0)
        pitch = pad_or_trim(pitch, target_len)
        pitch = normalize(pitch, self.data_parameters.get("pitch_mean", 0), self.data_parameters.get("pitch_std", 1))

        frames = mono.unfold(1, self.win_length, self.hop_length)
        energy = torch.sqrt(torch.mean(frames ** 2, dim=-1)).squeeze(0)
        energy = torch.nan_to_num(energy, nan=0.0, posinf=0.0, neginf=0.0)
        energy = pad_or_trim(energy, target_len)
        energy = normalize(energy, self.data_parameters.get("energy_mean", 0), self.data_parameters.get("energy_std", 1))
        return pitch, energy

    def get_text(self, text, add_blank=True):
        text_norm, cleaned_text = text_to_sequence(text, self.cleaners)
        if self.add_blank:
            text_norm = intersperse(text_norm, 0)
        text_norm = torch.IntTensor(text_norm)
        return text_norm, cleaned_text

    def get_durations(self, filepath, text):
        return None

    def __getitem__(self, index):
        return self.get_datapoint(self.filepaths_and_text[index])

    def __len__(self):
        return len(self.filepaths_and_text)


class TextMelBatchCollate:
    def __init__(self, n_spks):
        self.n_spks = n_spks

    def __call__(self, batch):
        B = len(batch)
        y_max_length = max([item["y"].shape[-1] for item in batch])
        y_max_length = fix_len_compatibility(y_max_length)
        x_max_length = max([item["x"].shape[-1] for item in batch])
        n_feats = batch[0]["y"].shape[-2]

        y = torch.zeros((B, n_feats, y_max_length), dtype=torch.float32)
        x = torch.zeros((B, x_max_length), dtype=torch.long)
        pitch = torch.zeros((B, y_max_length), dtype=torch.float32)
        energy = torch.zeros((B, y_max_length), dtype=torch.float32)
        y_lengths, x_lengths = [], []
        raw_texts = []
        filepaths = []

        for i, item in enumerate(batch):
            y_, x_ = item["y"], item["x"]
            pitch_, energy_ = item["pitch"], item["energy"]
            y_lengths.append(y_.shape[-1])
            x_lengths.append(x_.shape[-1])
            y[i, :, : y_.shape[-1]] = y_
            x[i, : x_.shape[-1]] = x_
            pitch[i, : pitch_.shape[-1]] = pitch_
            energy[i, : energy_.shape[-1]] = energy_
            raw_texts.append(item["raw_text"])
            filepaths.append(item["filepath"])

        y_lengths = torch.tensor(y_lengths, dtype=torch.long)
        x_lengths = torch.tensor(x_lengths, dtype=torch.long)

        return {
            "x": x,
            "x_lengths": x_lengths,
            "y": y,
            "y_lengths": y_lengths,
            "pitch": pitch,
            "energy": energy,
            "spks": None,
            "raw_texts": raw_texts,
            "durations": None,
        }
"""

with open("matcha/data/text_mel_datamodule.py", "w", encoding="utf-8") as f:
    f.write(datamodule_code)

print("✅ Đã cập nhật TextMelDataModule với pitch/energy prosody support!")

## 6. Xử lý và chuẩn bị Filelists

In [ ]:
# Xác định đường dẫn input files
AUDIO_DIR = "/kaggle/input/data-audio-ipa/kaggle/working/data/subs_add_con"
TRAIN_LIST_INPUT = "/kaggle/input/text-ipa/audio_text_train_filelist_new_ipa.txt"
VAL_LIST_INPUT = "/kaggle/input/text-ipa/audio_text_val_filelist_new_ipa.txt"

# Đường dẫn output (nơi lưu filelist đã xử lý)
TRAIN_LIST_OUTPUT = "/kaggle/working/fixed_train.txt"
VAL_LIST_OUTPUT = "/kaggle/working/fixed_val.txt"

print(f"Audio source: {AUDIO_DIR}")
print(f"Train list: {TRAIN_LIST_INPUT} -> {TRAIN_LIST_OUTPUT}")
print(f"Val list: {VAL_LIST_INPUT} -> {VAL_LIST_OUTPUT}")


## 7. Fix File Paths

In [ ]:
import os

def fix_file_paths(input_file, output_file, audio_folder_path):
    """Fix paths trong filelist để phù hợp với Kaggle environment"""
    with open(input_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    new_lines = []
    for line in lines:
        parts = line.strip().split('|')
        if len(parts) >= 3:
            # Tách filename từ path
            filename = parts[0].replace('\\', '/').split('/')[-1].replace('"', '')
            # Lấy text thường (cột 2)
            text = parts[1].replace('"', '').strip()
            # Lấy IPA (cột 3)
            ipa = parts[2].replace('"', '').strip()
            
            # Lưu: filename|text|ipa (TextMelDataModule sẽ tự nối với audio_root)
            new_lines.append(f"{filename}|{text}|{ipa}")
            
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write("\n".join(new_lines))
    
    print(f"✅ Processed {len(new_lines)} lines -> {output_file}")

# Fix cả train và val filelists
fix_file_paths(TRAIN_LIST_INPUT, TRAIN_LIST_OUTPUT, AUDIO_DIR)
fix_file_paths(VAL_LIST_INPUT, VAL_LIST_OUTPUT, AUDIO_DIR)


## 8. Tính Mel Statistics

In [ ]:
import torch
import torchaudio
from tqdm.auto import tqdm

# Cấu hình mel spectrogram
sample_rate = 22050
n_fft = 1024
n_mels = 80
hop_length = 256
win_length = 1024
f_min = 0
f_max = 8000

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=sample_rate, n_fft=n_fft, n_mels=n_mels, 
    hop_length=hop_length, win_length=win_length, 
    f_min=f_min, f_max=f_max, center=False, power=1.0, 
    norm="slaney", mel_scale="slaney"
)

mel_sum = 0
mel_sq_sum = 0
total_frames = 0

print("⏳ Đang tính toán Mel Statistics...")
with open(TRAIN_LIST_OUTPUT, 'r', encoding='utf-8') as f:
    lines = f.readlines()[:500]  # Lấy 500 files đầu để tính nhanh

for line in tqdm(lines):
    filename = line.split('|')[0]
    audio_path = os.path.join(AUDIO_DIR, filename)
    try:
        wav, sr = torchaudio.load(audio_path)
        if sr != sample_rate:
            wav = torchaudio.functional.resample(wav, sr, sample_rate)
        
        mel = mel_transform(wav)
        mel = torch.log(torch.clamp(mel, min=1e-5))
        
        mel_sum += torch.sum(mel)
        mel_sq_sum += torch.sum(mel ** 2)
        total_frames += mel.numel()
    except Exception as e:
        continue

# Tính Mean và Std
dataset_mean = (mel_sum / total_frames).item()
dataset_std = torch.sqrt((mel_sq_sum / total_frames) - (dataset_mean ** 2)).item()

print(f"\n✅ KẾT QUẢ MEL STATISTICS:")
print(f"MEAN: {dataset_mean:.4f}")
print(f"STD:  {dataset_std:.4f}")

# Lưu để dùng sau
CALCULATED_MEAN = dataset_mean
CALCULATED_STD = dataset_std


In [ ]:
# Tính nhanh pitch/energy statistics (subset 300 files để tiết kiệm thời gian)
import os
import torch
import torchaudio
from tqdm.auto import tqdm

# Đảm bảo các tham số đã được define (lấy từ cell trước)
if 'sample_rate' not in locals():
    sample_rate = 22050
if 'win_length' not in locals():
    win_length = 1024
if 'hop_length' not in locals():
    hop_length = 256

pitch_sum = pitch_sq_sum = 0.0
energy_sum = energy_sq_sum = 0.0
pitch_frames = energy_frames = 0

print("⏳ Đang tính Pitch/Energy Statistics trên 300 files đầu...")

with open(TRAIN_LIST_OUTPUT, 'r', encoding='utf-8') as f:
    lines = f.readlines()[:300]  # Giới hạn để nhanh hơn trên Kaggle

success_count = 0
error_count = 0

for idx, line in enumerate(tqdm(lines)):
    filename = line.split('|')[0].strip()
    audio_path = os.path.join(AUDIO_DIR, filename)
    
    try:
        if not os.path.exists(audio_path):
            error_count += 1
            continue
            
        wav, sr = torchaudio.load(audio_path)
        if sr != sample_rate:
            wav = torchaudio.functional.resample(wav, sr, sample_rate)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)

        # Pitch - sử dụng detect_pitch_frequency với API đúng (không có hop_length)
        # frame_time = hop_length / sample_rate để tương đương
        frame_time = hop_length / sample_rate  # ~0.0116s with hop=256, sr=22050
        pitch = torchaudio.functional.detect_pitch_frequency(
            wav, 
            sample_rate=sample_rate,
            frame_time=frame_time,
        ).squeeze(0)
        
        pitch = torch.log1p(pitch)
        pitch = torch.nan_to_num(pitch, nan=0.0, posinf=0.0, neginf=0.0)
        # Only count non-zero values
        pitch_nonzero = pitch[pitch > 0]
        if len(pitch_nonzero) > 0:
            pitch_sum += torch.sum(pitch_nonzero)
            pitch_sq_sum += torch.sum(pitch_nonzero ** 2)
            pitch_frames += len(pitch_nonzero)

        # Energy - tính RMS energy per frame
        frames = wav.unfold(1, win_length, hop_length)
        energy = torch.sqrt(torch.mean(frames ** 2, dim=-1)).squeeze(0)
        energy = torch.nan_to_num(energy, nan=0.0, posinf=0.0, neginf=0.0)
        # Only count non-zero values
        energy_nonzero = energy[energy > 0]
        if len(energy_nonzero) > 0:
            energy_sum += torch.sum(energy_nonzero)
            energy_sq_sum += torch.sum(energy_nonzero ** 2)
            energy_frames += len(energy_nonzero)
        
        success_count += 1
    except Exception as e:
        error_count += 1
        if idx < 3:
            print(f"[{idx}] Error: {e}")
        continue

print(f"\n✅ Đã xử lý thành công {success_count}/{len(lines)} files")
print(f"❌ Lỗi: {error_count} files")

if pitch_frames > 0:
    pitch_mean = (pitch_sum / pitch_frames).item()
    pitch_std = torch.sqrt((pitch_sq_sum / pitch_frames) - (pitch_mean ** 2)).item()
    print(f"✅ Pitch mean/std: {pitch_mean:.4f} / {pitch_std:.4f}")
else:
    print("⚠️ Không tính được pitch, dùng mặc định")
    pitch_mean, pitch_std = 0.0, 1.0

if energy_frames > 0:
    energy_mean = (energy_sum / energy_frames).item()
    energy_std = torch.sqrt((energy_sq_sum / energy_frames) - (energy_mean ** 2)).item()
    print(f"✅ Energy mean/std: {energy_mean:.4f} / {energy_std:.4f}")
else:
    print("⚠️ Không tính được energy, dùng mặc định")
    energy_mean, energy_std = 0.0, 1.0

PITCH_MEAN, PITCH_STD = pitch_mean, pitch_std
ENERGY_MEAN, ENERGY_STD = energy_mean, energy_std

## 9. Tạo Training Script

In [ ]:
import os

# Lấy Stats từ cell tính toán trước đó
try:
    my_mean = CALCULATED_MEAN
    my_std = CALCULATED_STD
    print(f"✅ Đã tìm thấy Mel Stats: Mean={my_mean:.4f}, Std={my_std:.4f}")
except NameError:
    print("⚠️ Không tìm thấy CALCULATED_MEAN. Dùng mel stats mặc định.")
    my_mean = -5.0
    my_std = 2.0

try:
    my_pitch_mean = PITCH_MEAN
    my_pitch_std = PITCH_STD
    my_energy_mean = ENERGY_MEAN
    my_energy_std = ENERGY_STD
    print(f"✅ Pitch Stats: Mean={my_pitch_mean:.4f}, Std={my_pitch_std:.4f}")
    print(f"✅ Energy Stats: Mean={my_energy_mean:.4f}, Std={my_energy_std:.4f}")
except NameError:
    print("⚠️ Không tìm thấy pitch/energy stats. Dùng mặc định 0/1.")
    my_pitch_mean, my_pitch_std = 0.0, 1.0
    my_energy_mean, my_energy_std = 0.0, 1.0

# Tạo training script với PROSODY SUPPORT ĐẦY ĐỦ
script_content = f"""
import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor, EarlyStopping
from lightning.pytorch.loggers import TensorBoardLogger
from matcha.models.matcha_tts import MatchaTTS
from matcha.data.text_mel_datamodule import TextMelDataModule
from matcha.text.symbols import symbols
from argparse import Namespace
import argparse
import torch.serialization

# Fix pickle
torch.serialization.add_safe_globals([argparse.Namespace])

# CONFIG TRAINING VỚI PROSODY
CONFIG = {{
    "train_filelist": "{TRAIN_LIST_OUTPUT}",
    "val_filelist": "{VAL_LIST_OUTPUT}",
    "output_dir": "outputs/matcha_prosody",
    
    # Model
    "n_spks": 1,
    "spk_emb_dim": 64,
    "n_feats": 80,
    
    # ✅ PROSODY CONFIG - IMPORTANT!
    "llm_model_name": "vinai/phobert-base",  # PhoBERT cho Vietnamese
    "prosody_dim": 256,  # Dimension của prosody embeddings từ PhoBERT
    "use_token_level_prosody": True,  # NEW: Token-level prosody alignment
    "finetune_llm": True,  # NEW: Fine-tune PhoBERT (set True cho GPU mạnh >= 24GB)
    
    # Training
    "batch_size": 16,
    "learning_rate": 1e-4,
    "max_epochs": 100,
    "num_workers": 2,
    
    # Hardware
    "accelerator": "gpu",
    "devices": 2,
    "strategy": "ddp",
    "precision": "32-true",
    
    # Paths
    "audio_root": "{AUDIO_DIR}",
    "resume_from_checkpoint": None,
}}

CONFIG["n_vocab"] = len(symbols) + 1
DATA_STATS = {{
    "mel_mean": {my_mean:.4f}, "mel_std": {my_std:.4f},
    "pitch_mean": {my_pitch_mean:.4f}, "pitch_std": {my_pitch_std:.4f},
    "energy_mean": {my_energy_mean:.4f}, "energy_std": {my_energy_std:.4f},
}}

print("="*80)
print("🍵 MATCHA-TTS TRAINING WITH LLM PROSODY ANALYSIS")
print("="*80)
print(f"PhoBERT Model: {{CONFIG['llm_model_name']}}")
print(f"Prosody Dimension: {{CONFIG['prosody_dim']}}")
print(f"Mel Stats: Mean={{DATA_STATS['mel_mean']:.4f}}, Std={{DATA_STATS['mel_std']:.4f}}")
print(f"Pitch Stats: Mean={{DATA_STATS['pitch_mean']:.4f}}, Std={{DATA_STATS['pitch_std']:.4f}}")
print(f"Energy Stats: Mean={{DATA_STATS['energy_mean']:.4f}}, Std={{DATA_STATS['energy_std']:.4f}}")
print("="*80)

def create_model():
    '''Tạo MatchaTTS model với PROSODY SUPPORT'''
    
    encoder_params = {{
        "n_feats": 80, "n_channels": 192, "filter_channels": 768, 
        "filter_channels_dp": 256, "n_heads": 2, "n_layers": 6, 
        "kernel_size": 3, "p_dropout": 0.1, "spk_emb_dim": 64, 
        "n_spks": 1, "prenet": True
    }}
    
    # ✅ FIX: Duration predictor params (required by TextEncoder)
    duration_predictor_params = {{
        "filter_channels_dp": 256,
        "kernel_size": 3,
        "p_dropout": 0.1,
    }}
    
    decoder_params = {{
        "channels": (256, 256), 
        "dropout": 0.05,
        "attention_head_dim": 64,
        "n_blocks": 1,
        "num_mid_blocks": 2,
        "num_heads": 4,
        "act_fn": "snakebeta",
    }}
    
    # ✅ FIX: CFM params MUST include 'solver' (euler/midpoint/heun/etc)
    cfm_params = {{
        "n_feats": 80, "channels": 192, "dropout": 0.1, "n_layers": 6, 
        "n_heads": 2, "window_size": 4, "hidden_size": 192,
        "solver": "euler",  # ODE solver: euler/midpoint/heun
        "sigma_min": 1e-4,  # Min noise level
    }}
    
    # Optimizer: Sử dụng import torch.optim để có class reference
    optimizer_kwargs = {{"lr": CONFIG["learning_rate"], "betas": (0.9, 0.98), "eps": 1e-9}}

    model = MatchaTTS(
        n_vocab=CONFIG["n_vocab"],
        n_spks=CONFIG["n_spks"],
        spk_emb_dim=CONFIG["spk_emb_dim"],
        n_feats=CONFIG["n_feats"],
        encoder=Namespace(
            encoder_type="fftransformer", 
            encoder_params=Namespace(**encoder_params), 
            duration_predictor_params=Namespace(**duration_predictor_params)
        ),
        decoder=decoder_params,
        cfm=Namespace(**cfm_params),
        data_statistics=DATA_STATS,
        out_size=128,
        optimizer=torch.optim.Adam,
        optimizer_kwargs=optimizer_kwargs,
        scheduler=None,
        prior_loss=True,
        use_precomputed_durations=False,
        use_token_level_prosody=CONFIG.get("use_token_level_prosody", True),
        finetune_llm=CONFIG.get("finetune_llm", True),
    )
    return model


def create_datamodule():
    datamodule = TextMelDataModule(
        name="matcha_vi",
        train_filelist_path=CONFIG["train_filelist"],
        valid_filelist_path=CONFIG["val_filelist"],
        batch_size=CONFIG["batch_size"],
        num_workers=CONFIG["num_workers"],
        pin_memory=True,
        cleaners=["basic_cleaners_phothong"],
        add_blank=True,
        n_spks=CONFIG["n_spks"],
        n_fft=1024,
        n_feats=80,
        sample_rate=22050,
        hop_length=256,
        win_length=1024,
        f_min=0,
        f_max=8000,
        data_statistics=DATA_STATS,
        seed=42,
        load_durations=False,
        audio_root=CONFIG["audio_root"],
    )
    return datamodule


def train():
    # 1. Init
    model = create_model()
    datamodule = create_datamodule()

    # 2. Callbacks
    checkpoint_cb = ModelCheckpoint(
        dirpath=CONFIG["output_dir"],
        filename="matcha-prosody-{{epoch:02d}}-{{step:04d}}", 
        save_top_k=3, 
        monitor="loss/val_epoch",
        save_last=True
    )
    lr_monitor = LearningRateMonitor(logging_interval="step")

    # 3. Trainer
    trainer = pl.Trainer(
        accelerator=CONFIG["accelerator"], 
        devices=CONFIG["devices"], 
        strategy=CONFIG["strategy"], 
        max_epochs=CONFIG["max_epochs"],
        callbacks=[checkpoint_cb, lr_monitor], 
        logger=TensorBoardLogger("outputs", name="logs"),
        precision=CONFIG["precision"],
        gradient_clip_val=1.0
    )

    # 4. Train
    trainer.fit(model, datamodule=datamodule)
    
    print("\\\\n" + "="*80)
    print("✅ TRAINING COMPLETED!")
    print("="*80)
    print("Best checkpoint: " + str(checkpoint_cb.best_model_path))
    print("Output dir: " + CONFIG["output_dir"])
    print("="*80)

if __name__ == "__main__":
    train()
"""

# Save script - TẠO Ở THƯ MỤC HIỆN TẠI (không tạo subfolder)
with open("train_matcha_final.py", "w", encoding="utf-8") as f:
    f.write(script_content)

print("✅ Đã tạo training script: train_matcha_final.py")
print("📝 Script đã cấu hình:")
print(f"   - PhoBERT: vinai/phobert-base")
print(f"   - Prosody Dim: 256")
print(f"   - Token-Level Prosody: True (mỗi phoneme có prosody riêng)")
print(f"   - Fine-tune PhoBERT: False (đóng băng để tiết kiệm GPU)")
print(f"   - Batch Size: 16 (tăng từ 8, tối ưu cho 2 GPU)")
print(f"   - Learning Rate: 1e-4")
print(f"   - Max Epochs: 20")
print(f"   - GPU: 2 devices with ddp")
print(f"   - Mel: Mean={my_mean:.4f}, Std={my_std:.4f}")
print(f"   - Pitch: Mean={my_pitch_mean:.4f}, Std={my_pitch_std:.4f}")
print(f"   - Energy: Mean={my_energy_mean:.4f}, Std={my_energy_std:.4f}")
print("\\n🎯 Cải tiến được áp dụng:")
print("   ✅ Token-level prosody alignment (mỗi phoneme có vector riêng)")
print("   ✅ Pause predictor (dự đoán ngắt nghỉ)")
print("   ✅ Boundary detector (phát hiện ranh giới cụm từ)")
print("   ⚠️  PhoBERT fine-tuning tắt (bật nếu có GPU >= 24GB)")


print(f"   - Energy: Mean={my_energy_mean:.4f}, Std={my_energy_std:.4f}")print(f"   - Energy: Mean={my_energy_mean:.4f}, Std={my_energy_std:.4f}")


## 🎯 Prosody Technique Được Áp Dụng

### 📋 Architecture:
```
Raw Vietnamese Text (Column 2)
        ↓
   [PhoBERT]  ← Lấy embeddings từ LLM
        ↓
Prosody Features (768-dim)
        ↓
[Projection Layer] → prosody_dim (256)
        ↓
[Prosody Fusion Module]
        ↓
Kết hợp với Text Encoder Output
        ↓
[CFM Decoder] ← Mel-spectrogram cuối cùng
```

### ✨ Key Components:

1. **LLMProsodyAnalyzer** (`matcha/models/components/prosody_analyzer.py`)
   - Dùng PhoBERT (vinai/phobert-base) để phân tích prosody
   - Extract toàn bộ 768-dim embeddings từ input text
   - Frozen LLM weights (không train)
   - Project xuống prosody_dim (256)

2. **ProsodyFusion** (`matcha/models/components/prosody_fusion.py`)
   - Cross-attention fusion giữa text encoder output và prosody features
   - Gating mechanism để kiểm soát ảnh hưởng của prosody
   - Output dimension = text encoder dimension

3. **Data Pipeline**
   - Input filelist: `filename|raw_text|ipa_text`
   - raw_text → PhoBERT → prosody embeddings
   - ipa_text → mel-spectrogram
   - Kết hợp cả hai qua fusion module

4. **Training Strategy**
   - Batch có chứa `raw_texts` để extract prosody on-the-fly
   - PhoBERT frozen, chỉ train fusion module + decoder
   - Joint learning của text synthesis + prosody conditioning

### 📊 Configuration:
- llm_model_name: `vinai/phobert-base`
- prosody_dim: 256 (đủ lớn để capture prosody features)
- fusion_type: attention-based với gating
- Prosody influence: controlled bằng learned gate

In [ ]:
# ============================================================================
# 📝 PROSODY USAGE EXAMPLES
# ============================================================================

print("""
🎯 PROSODY TECHNIQUE DETAILS:
================================

1. **Filelist Format (3 columns):**
   filename|raw_vietnamese_text|ipa_phonemes
   
   Example:
   audio_001.wav|xin chào các bạn|ks i n tʃ ə w ə ʔ i ə...
   
   Column 1: Filename (relative path)
   Column 2: RAW VIETNAMESE TEXT → input cho PhoBERT
   Column 3: IPA PHONEMES → input cho Matcha synthesis

2. **PhoBERT Prosody Analysis:**
   - Tokenize raw_text với PhoBERT tokenizer
   - Pass qua PhoBERT model → 768-dim embeddings
   - Project xuống 256-dim prosody_dim
   - Fusion với text encoder output qua attention

3. **Training Data Requirements:**
   ✓ Raw Vietnamese text MUST be provided (column 2)
   ✓ Text should be grammatically correct
   ✓ PhoBERT sẽ extract ngữ cảnh + prosody patterns
   
4. **Inference with Prosody:**
   model.synthesise(
       x=ipa_tokens,              # IPA tokenized
       x_lengths=seq_lengths,     # Token sequence lengths
       raw_texts=["xin chào"],    # RAW TEXT for prosody!
       n_timesteps=10             # Diffusion steps
   )

5. **Performance Impact:**
   - Adding prosody increases VRAM ~5% (768 embeddings/batch)
   - Inference ~10% slower (PhoBERT forward pass)
   - Quality improvement: ✓✓✓ (better intonation, stress, rhythm)

💡 TIP: Để tốt nhất, chuẩn bị raw_text sao cho:
   - Chính tả đúng (spelling matters for PhoBERT)
   - Dấu câu chính xác (periods, commas affect prosody)
   - Không viết tắt
""")


## 10. BẮT ĐẦU TRAINING

In [ ]:
# Cài đặt rootutils (required by matcha/train.py)
!pip install rootutils -q
print("✅ Đã cài rootutils!")


In [ ]:
!python train_matcha_final.py

## 11. Kiểm tra text processing

In [ ]:
from matcha.text import cleaners
from matcha.text import symbols

print("🔤 Tổng số ký hiệu trong tokenizer:", len(symbols))
print(symbols[:50])

In [ ]:
# Utility: Inspect text length vs model limits and suggest chunking
import re
from typing import List, Tuple

from matcha.text import text_to_sequence
from matcha.text.cleaners import basic_cleaners_phothong

# Optional: check PhoBERT tokenizer limits if available
try:
    from transformers import AutoTokenizer, AutoModel
    _tok = AutoTokenizer.from_pretrained("vinai/phobert-base")
    _llm = AutoModel.from_pretrained("vinai/phobert-base")
    _special_tokens = _tok.num_special_tokens_to_add(pair=False)
    _llm_max_pos = getattr(_llm.config, "max_position_embeddings", 512)
    _llm_pos_actual = getattr(getattr(_llm, "embeddings", None), "position_embeddings", None)
    if _llm_pos_actual is not None:
        _llm_max_safe = max(1, _llm_pos_actual.num_embeddings - 1)
    else:
        _llm_max_safe = _llm_max_pos
except Exception:
    _tok = None
    _llm = None
    _special_tokens = 2
    _llm_max_safe = 512


def _sentence_split(text: str) -> List[str]:
    parts = re.split(r"([.!?])", text)
    # Re-attach punctuation
    sents = ["".join(parts[i:i+2]).strip() for i in range(0, len(parts), 2)]
    return [s for s in sents if s]


def inspect_text(text: str, phoneme_warn: int = 320) -> Tuple[int, int]:
    # Clean & phonemize to estimate phoneme sequence length
    cleaned = basic_cleaners_phothong(text)
    seq, _ = text_to_sequence(text, ["basic_cleaners_phothong"])  # uses cleaners internally
    phoneme_len = len(seq)

    # Tokenizer length (raw text) if available
    if _tok is not None:
        enc = _tok(text, truncation=False, add_special_tokens=True, return_tensors=None)
        llm_len = len(enc["input_ids"])  # includes specials
    else:
        llm_len = 0

    print(f"🔎 Phoneme length: {phoneme_len}")
    if _tok is not None:
        print(f"🔎 PhoBERT tokens (with specials): {llm_len} | safe max: {_llm_max_safe}")
    else:
        print("ℹ️ PhoBERT tokenizer unavailable; skipping LLM length check.")

    if phoneme_len > phoneme_warn:
        print("⚠️ Phoneme sequence is long; risk of high VRAM/OOM. Consider chunking.")
    if _tok is not None and llm_len > _llm_max_safe:
        print("⚠️ Raw text exceeds PhoBERT position limit; it will be truncated.")

    return phoneme_len, llm_len


def suggest_chunking(text: str) -> List[str]:
    sents = _sentence_split(text)
    if len(sents) <= 1:
        # Fallback: split by commas if no clear sentences
        sents = [t.strip() for t in re.split(r",", text) if t.strip()]
    print(f"🧩 Suggested {len(sents)} chunks for synthesis.")
    return sents

# Demo: replace with your paragraph
sample_text = "Xin chào các bạn, đây là một đoạn văn dài thử nghiệm để đánh giá khả năng xử lý văn bản của hệ thống. Hệ thống sẽ kiểm tra độ dài phoneme và giới hạn của PhoBERT."
inspect_text(sample_text)
chunks = suggest_chunking(sample_text)
for i, c in enumerate(chunks[:3]):
    print(f"{i+1:02d}. {c}")

## 13. Generate data statistics

In [ ]:
# Generate data statistics - sử dụng stats đã tính từ cell 8 và 9
print("="*80)
print("📊 DATA STATISTICS (từ cell 8 và 9)")
print("="*80)
print(f"✅ Mel Statistics:")
print(f"   Mean: {CALCULATED_MEAN:.4f}")
print(f"   Std:  {CALCULATED_STD:.4f}")
print(f"\n✅ Pitch Statistics:")
print(f"   Mean: {PITCH_MEAN:.4f}")
print(f"   Std:  {PITCH_STD:.4f}")
print(f"\n✅ Energy Statistics:")
print(f"   Mean: {ENERGY_MEAN:.4f}")
print(f"   Std:  {ENERGY_STD:.4f}")
print("="*80)
print("\n📝 Các stats này sẽ được dùng trong training script (cell 9)")


## 14. Kiểm tra symbols trong filelist

In [ ]:
# Kiểm tra symbols trong filelist
from matcha.text import symbols

bad_lines = []
print(f"🔍 Đang kiểm tra symbols trong filelist: {TRAIN_LIST_OUTPUT}")
print(f"📊 Tổng số symbols có sẵn: {len(symbols)}")

with open(TRAIN_LIST_OUTPUT, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        try:
            parts = line.strip().split("|")
            if len(parts) < 3:
                print(f"❌ Dòng {i}: format sai (cần 3 cột), lỗi: {line[:50]}")
                continue
            path, raw_text, ipa_text = parts[0], parts[1], parts[2]
        except ValueError as e:
            print(f"❌ Dòng {i}: lỗi parse: {e}")
            continue

        # Kiểm tra IPA text (cột 3)
        for c in ipa_text:
            if c not in symbols:
                bad_lines.append((i, c, ipa_text))
                break

if bad_lines:
    print(f"\n⚠️ Tìm thấy {len(bad_lines)} dòng có ký tự không hợp lệ:")
    for i, c, text in bad_lines[:10]:
        print(f"   Dòng {i}: ký tự '{c}' (ord={ord(c)}) → {text[:60]}")
    if len(bad_lines) > 10:
        print(f"   ... và {len(bad_lines) - 10} dòng khác")
else:
    print(f"\n✅ Tất cả ký tự trong filelist đều hợp lệ!")
    
print(f"\n✅ Hoàn tất kiểm tra {i} dòng")


## 16. Xem TensorBoard

In [ ]:
%load_ext tensorboard

# ✅ FIX: Update path to match training script output
# Training script saves logs to: /kaggle/working/IPIATTS/outputs/logs/
%tensorboard --logdir /kaggle/working/IPIATTS/outputs/logs --port 6006

## 17. Phân tích training loss

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from tensorboard.backend.event_processing import event_accumulator
import os

# ✅ FIX: Update đường dẫn TensorBoard logs cho đúng với training script
# Training script lưu ở: outputs/logs/
log_dir = "/kaggle/working/IPIATTS/outputs/logs"

# Tìm version folder (version_0, version_1, etc.)
if os.path.exists(log_dir):
    versions = [d for d in os.listdir(log_dir) if d.startswith("version_")]
    if versions:
        # Lấy version mới nhất
        latest_version = sorted(versions)[-1]
        tensorboard_path = os.path.join(log_dir, latest_version)
        print(f"📂 Sử dụng logs từ: {tensorboard_path}")
    else:
        print("⚠️ Không tìm thấy version folder trong logs!")
        tensorboard_path = log_dir
else:
    print(f"❌ Không tìm thấy log directory: {log_dir}")
    print("💡 Đảm bảo đã chạy training (cell 10) trước khi xem logs!")
    tensorboard_path = None

if tensorboard_path and os.path.exists(tensorboard_path):
    try:
        ea = event_accumulator.EventAccumulator(tensorboard_path)
        ea.Reload()

        # Lấy dữ liệu loss
        scalars = ea.Scalars('loss/train_step')
        steps = [s.step for s in scalars]
        values = [s.value for s in scalars]

        # Vẽ đồ thị
        plt.figure(figsize=(10,5))
        plt.plot(steps, values, label='Train Loss', linewidth=1.5)
        plt.xlabel('Step')
        plt.ylabel('Loss')
        plt.title('Biểu đồ Loss theo Step (Matcha-TTS)')
        plt.legend()
        plt.grid(True)
        plt.show()

        print(f"✅ Loss cuối cùng: {scalars[-1].value:.4f}")
        print(f"📊 Trung bình 10 step cuối: {np.mean([s.value for s in scalars[-10:]]):.4f}")
    except Exception as e:
        print(f"❌ Lỗi khi đọc TensorBoard logs: {e}")
        print("💡 Có thể training chưa chạy hoặc chưa có đủ dữ liệu")
else:
    print("⏭️ Skip cell này - chạy lại sau khi training xong!")

## 20. Utilities và Helper Functions

In [ ]:
# Kiểm tra checkpoint
def check_checkpoint(ckpt_path):
    import torch
    ckpt = torch.load(ckpt_path, map_location='cpu')
    print("Checkpoint keys:", ckpt.keys())
    if 'epoch' in ckpt:
        print(f"Epoch: {ckpt['epoch']}")
    if 'global_step' in ckpt:
        print(f"Global step: {ckpt['global_step']}")
    return ckpt

# Tính toán mel statistics
def calculate_mel_stats(filelist_path, audio_root):
    import librosa
    import numpy as np
    from tqdm import tqdm
    
    mels = []
    with open(filelist_path, 'r', encoding='utf-8') as f:
        for line in tqdm(f.readlines()[:100]):  # Sample 100 files
            audio_path = line.strip().split('|')[0]
            y, sr = librosa.load(audio_path, sr=22050)
            mel = librosa.feature.melspectrogram(
                y=y, sr=sr, n_fft=1024, hop_length=256, 
                win_length=1024, n_mels=80, fmin=0, fmax=8000
            )
            mel_db = librosa.power_to_db(mel, ref=np.max)
            mels.append(mel_db)
    
    mels = np.concatenate(mels, axis=1)
    mel_mean = np.mean(mels)
    mel_std = np.std(mels)
    
    print(f"Mel Mean: {mel_mean}")
    print(f"Mel Std: {mel_std}")
    return mel_mean, mel_std

## 21. Find và list checkpoints

In [ ]:
# ✅ Tìm tất cả checkpoints đã được tạo
!find /kaggle/working/IPIATTS/outputs -type f -name "*.ckpt" 2>/dev/null | head -20

## 22. Test inference sau khi train

In [ ]:
# Load model từ checkpoint
from matcha.models.matcha_tts import MatchaTTS
import torch
import torch.serialization
import torch.optim
import torch.nn
import argparse
import os

# ✅ FIX: Allowlist all necessary classes cho PyTorch 2.6+ (required for unpickling checkpoint)
torch.serialization.add_safe_globals([
    argparse.Namespace,
    torch.optim.Adam,
])

# Danh sách checkpoint có thể tồn tại (ưu tiên theo thứ tự)
candidate_checkpoints = [
    "/kaggle/working/IPIATTS/outputs/matcha_prosody/last.ckpt",
    "/kaggle/working/IPIATTS/outputs/matcha_prosody/matcha-prosody-epoch=08-step=5354.ckpt",
    "/kaggle/working/IPIATTS/outputs/matcha_prosody/matcha-prosody-epoch=07-step=4759.ckpt",
    "/kaggle/working/IPIATTS/outputs/matcha_prosody/matcha-prosody-epoch=05-step=3569.ckpt",
    "/kaggle/working/IPIATTS/outputs/matcha_prosody/last-v1.ckpt",
    "/kaggle/working/IPIATTS/outputs/matcha_prosody/matcha-prosody-epoch=00-step=0594.ckpt",
    "/kaggle/working/IPIATTS/IPIATTS/outputs/matcha_prosody/last.ckpt",
]

# ✅ Chọn checkpoint đầu tiên tìm thấy
checkpoint_path = next((p for p in candidate_checkpoints if os.path.exists(p)), None)

if checkpoint_path is None:
    print("❌ Không tìm thấy checkpoint trong các đường dẫn candidate.")
    print("\n💡 Chạy cell 40 để tìm checkpoints có sẵn, hoặc thử thêm đường dẫn.")
    model = None
else:
    print(f"📥 Loading checkpoint: {checkpoint_path}")
    model = MatchaTTS.load_from_checkpoint(checkpoint_path)
    model.eval()
    model = model.cuda() if torch.cuda.is_available() else model
    print("✅ Model loaded successfully!")
    print(f"📁 Checkpoint: {checkpoint_path}")

In [ ]:
# Synthesize speech với HiFi-GAN vocoder
from matcha.text import text_to_sequence
from matcha.utils.utils import intersperse
import soundfile as sf
import sys
import os
import torch
from argparse import Namespace

# Đảm bảo model đã được load
if "model" not in globals() or model is None:
    raise RuntimeError("Model chưa được load. Chạy cell 41 và đảm bảo checkpoint tồn tại.")

# Add HiFi-GAN to path
sys.path.append('/kaggle/working/IPIATTS/matcha/hifigan')
from matcha.hifigan.models import Generator
from matcha.hifigan.config import v1

# ===== BƯỚC 1: TẠO MEL-SPECTROGRAM TỪ TEXT =====
text = "xin chào các bạn"
print(f"📝 Text input: {text}")

x = torch.tensor(
    intersperse(text_to_sequence(text, ["basic_cleaners_phothong"])[0], 0),
    dtype=torch.long
)[None]

# ✅ FIX: Add x_lengths parameter (required by synthesise method)
x_lengths = torch.LongTensor([x.shape[-1]])

if torch.cuda.is_available():
    x = x.cuda()
    x_lengths = x_lengths.cuda()

print("🔊 Generating mel-spectrogram...")
with torch.no_grad():
    output = model.synthesise(x, x_lengths, n_timesteps=10)
    mel = output['mel']
    
print(f"✅ Generated mel shape: {mel.shape}")

# ===== BƯỚC 2: LOAD HIFI-GAN VOCODER =====
print("\n📦 Loading HiFi-GAN vocoder...")
hifigan_checkpoint = "/kaggle/working/IPIATTS/matcha/hifigan/checkpoints/g_02500000"

# ✅ FIX: Convert dict to Namespace so Generator can access attributes
hifigan_config = Namespace(**v1) if isinstance(v1, dict) else v1

# Create HiFi-GAN generator
vocoder = Generator(hifigan_config)
state_dict_g = torch.load(hifigan_checkpoint, map_location='cpu')
vocoder.load_state_dict(state_dict_g['generator'])
vocoder.eval()
vocoder.remove_weight_norm()

if torch.cuda.is_available():
    vocoder = vocoder.cuda()
    
print("✅ HiFi-GAN loaded!")

# ===== BƯỚC 3: CONVERT MEL → AUDIO =====
print("\n🎵 Converting mel to audio...")
with torch.no_grad():
    # ✅ FIX: Try to get data_statistics from model's hparams, otherwise use stats from training
    try:
        mel_mean = model.hparams.data_statistics['mel_mean']
        mel_std = model.hparams.data_statistics['mel_std']
        print(f"   Using model's mel stats: mean={mel_mean:.4f}, std={mel_std:.4f}")
        mel_denorm = mel * mel_std + mel_mean
    except:
        # If stats not available, try using global variables from training cells
        try:
            mel_mean = CALCULATED_MEAN
            mel_std = CALCULATED_STD
            print(f"   Using training mel stats: mean={mel_mean:.4f}, std={mel_std:.4f}")
            mel_denorm = mel * mel_std + mel_mean
        except:
            # Last resort: use mel as-is (model might output denormalized mel already)
            print("   ⚠️ No mel stats found - using mel as-is")
            mel_denorm = mel
    
    # Generate audio
    audio = vocoder(mel_denorm).squeeze()
    
    if torch.cuda.is_available():
        audio = audio.cpu()
    
    audio = audio.numpy()

print(f"✅ Generated audio shape: {audio.shape}")
print(f"   Duration: {len(audio) / 22050:.2f} seconds")

# ===== BƯỚC 4: SAVE AUDIO FILE =====
output_path = "/kaggle/working/synthesized_audio.wav"
sf.write(output_path, audio, 22050)
print(f"\n💾 Saved audio to: {output_path}")

# ===== BƯỚC 5: PLAY AUDIO IN NOTEBOOK =====
from IPython.display import Audio, display
print("\n🔊 Playing audio:")
display(Audio(audio, rate=22050))